# Prototype filter & channel-to-frequency mapping

This notebook tests two things in `pypoly.channelizer`:

1. `design_prototype_filter` produces a sane low-pass prototype.
2. `PolyphaseAnalysisChannelizer` routes each input frequency to the *correct* output channel.

Point (2) was previously broken: the analysis combiner used `np.fft.fft` where it needed `np.fft.ifft`
to match the polyphase commutator's sample-distribution direction (`x[nM - p]`). The effect was that a
tone at channel `k` showed up in output bin `(M - k) % M` instead of bin `k` (DC was unaffected, which is
why a quick DC-only smoke test wouldn't have caught it). This notebook demonstrates the corrected mapping.

In [ ]:
import sys
from pathlib import Path

_src = Path.cwd().parent / "src"
if _src.exists() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import freqz

from pypoly import PolyphaseAnalysisChannelizer, design_prototype_filter

%matplotlib inline

## Prototype filter response

Design an `M = 8` channel prototype and look at its magnitude response. We expect a low-pass shape with
the -6 dB point near `pi / M` (one channel's worth of bandwidth) and a stopband governed by the Kaiser window.

In [ ]:
M = 8
TAPS_PER_CHANNEL = 16

h = design_prototype_filter(num_channels=M, taps_per_channel=TAPS_PER_CHANNEL)
print(f"num taps: {h.size} (expected {M * TAPS_PER_CHANNEL})")

w, H = freqz(h, worN=4096)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(w / np.pi, 20 * np.log10(np.maximum(np.abs(H), 1e-12)))
ax.axvline(1 / M, color="k", linestyle="--", linewidth=1, label="pi / M (channel edge)")
ax.set_xlabel("Normalized frequency (x pi rad/sample)")
ax.set_ylabel("Magnitude (dB)")
ax.set_title("Prototype low-pass filter response")
ax.set_ylim(-100, 5)
ax.legend()
ax.grid(True, alpha=0.3)

## Channel-to-frequency mapping

Feed a pure complex tone at each of the `M` channel center frequencies `k0 * fs / M` into the analysis
channelizer, then measure steady-state energy in every output channel. A correct implementation should put
(almost) all of the energy for input tone `k0` into output channel `k0`, i.e. an identity matrix.

In [ ]:
analysis = PolyphaseAnalysisChannelizer.from_design(num_channels=M, taps_per_channel=TAPS_PER_CHANNEL)

n = np.arange(4096)
energy_matrix = np.zeros((M, M))

for k0 in range(M):
    x = np.exp(1j * 2 * np.pi * k0 * n / M)
    y = analysis.process(x)
    energy = np.mean(np.abs(y[:, 200:]) ** 2, axis=1)  # skip filter transient
    energy_matrix[k0] = energy / energy.max()

winners = energy_matrix.argmax(axis=1)
print("input channel -> output channel:", dict(enumerate(winners)))
assert np.array_equal(winners, np.arange(M)), "channel mapping is not the identity!"

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(energy_matrix, cmap="viridis", vmin=0, vmax=1)
ax.set_xlabel("output channel")
ax.set_ylabel("input tone channel (k0)")
ax.set_title("Normalized output energy per input tone\n(should be the identity matrix)")
ax.set_xticks(range(M))
ax.set_yticks(range(M))
fig.colorbar(im, ax=ax, label="normalized energy")

## Per-channel passband shape

Sweep a tone continuously across the full normalized band (not just the bin centers) and record the energy
captured by every output channel. This is the kind of "channel selectivity" plot shown in the MATLAB DSP
System Toolbox channelizer examples: each channel should show a passband centered on its own bin and
rejection elsewhere.

In [ ]:
freqs_norm = np.linspace(-0.5, 0.5, 400)  # cycles/sample, full Nyquist range
n = np.arange(4096)
sweep_energy = np.zeros((len(freqs_norm), M))

for i, f in enumerate(freqs_norm):
    x = np.exp(1j * 2 * np.pi * f * n)
    y = analysis.process(x)
    sweep_energy[i] = np.mean(np.abs(y[:, 200:]) ** 2, axis=1)

sweep_db = 10 * np.log10(np.maximum(sweep_energy, 1e-12) / sweep_energy.max())

fig, ax = plt.subplots(figsize=(8, 4))
for k in range(M):
    ax.plot(freqs_norm, sweep_db[:, k], label=f"ch {k}")
ax.set_xlabel("Input frequency (cycles/sample)")
ax.set_ylabel("Channel output energy (dB)")
ax.set_title("Per-channel selectivity across the full band")
ax.set_ylim(-80, 5)
ax.legend(ncol=4, fontsize=8)
ax.grid(True, alpha=0.3)